## NHIES Analysis
### Point 3 - HH Consumed Food Item (%)
### 18 December 2020

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 220)
pd.set_option("display.max_columns", None)

In [2]:
# Read in DRB
fn = 'Diary_coicopV12.dta'
path = Path.cwd().joinpath('data', 'drb', fn)

# Read in full 7-day diary dataset
df = pd.read_stata(path)

In [3]:
# Convert category to string
cat_cols = ['region', 'purch_cons', 'coicop', 'source', 'unit']

df[cat_cols] = df[cat_cols].astype(str)

# Sort values by region
df.sort_values(by='region', inplace=True)

In [4]:
def split_coicop_column(df):
    # Clean whitespace from Food_Item
    df.loc[:, 'coicop'] = df.coicop.str.strip()

    # Create separate values for Food_Item column
    df['COICOP'] = df['coicop'].str.split(" ", 1).str.get(0)
    df['Label'] = df['coicop'].str.split(" ", 1).str.get(1)

    return df

In [5]:
df = split_coicop_column(df)

In [6]:
# Clean whitespace from Label
df.loc[:, 'Label'] = df.Label.str.strip()
# Clean whitespace from COICOP
df.loc[:, 'COICOP'] = df.COICOP.str.strip()

In [7]:
# Classification of fortification vehicles
wheat_items = ['Bread (white, brown, whole wheat, rye, maize, etc.)', 'Brotchen', 'Biscuits, rusks', 'Vetkoek', 'Macaroni, spaghetti, noodles', 'Cakes (all types)', 'Pies & pizzas', 'Bread/ cake flour (all types)']

maize_items = ['Traditional bread, ash bread, oshikwiila, oshima, omungome', 'Maize meal/ grain/ samp']

salt_items = ['Salt']

oil_items = ['Cooking oil']

sugar_items = ['Sugar']

In [8]:
df['wheat_YN'] = np.where(df['Label'].isin(wheat_items), 1, 0)
df['maize_YN'] = np.where(df['Label'].isin(maize_items), 1, 0)
df['salt_YN'] = np.where(df['Label'].isin(salt_items), 1, 0)
df['oil_YN'] = np.where(df['Label'].isin(oil_items), 1, 0)
df['sugar_YN'] = np.where(df['Label'].isin(sugar_items), 1, 0)

## Agrgegate by hhid

In [9]:
df_sum = df.groupby('hhid').agg(sum)

In [10]:
df_sum['wheat_YN'] = np.where(df_sum['wheat_YN'] > 1, 1, df_sum['wheat_YN'])
df_sum['maize_YN'] = np.where(df_sum['maize_YN'] > 1, 1, df_sum['maize_YN'])
df_sum['salt_YN'] = np.where(df_sum['salt_YN'] > 1, 1, df_sum['salt_YN'])
df_sum['oil_YN'] = np.where(df_sum['oil_YN'] > 1, 1, df_sum['oil_YN'])
df_sum['sugar_YN'] = np.where(df_sum['sugar_YN'] > 1, 1, df_sum['sugar_YN'])

In [11]:
df_sum.reset_index(inplace=True)

In [12]:
food_cols = ['wheat_YN', 'maize_YN', 'salt_YN', 'oil_YN', 'sugar_YN']
item_mean = pd.DataFrame(df_sum[food_cols].mean(axis=0))

In [13]:
fn = 'item_mean.csv'
path = Path.cwd().joinpath('output', fn)
item_mean.to_csv(path)